In [1]:
import sys
sys.path.append("..")

from pathlib import Path
import pandas as pd
from turbofan.data import load_cmapss

In [2]:
RAW = Path("../data/raw")
path = Path(RAW / "test_FD001.txt")

In [3]:
test = load_cmapss(path)
test.sample(2)

,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
10678,82,78,0.0038,0.0005,100.0,518.67,642.50,1589.65,1404.38,14.62,...,521.37,2388.08,8142.61,8.4396,0.03,393,2388,100.0,39.04,23.3646
9767,76,53,0.0006,0.0000,100.0,518.67,642.04,1587.57,1399.08,14.62,...,522.15,2388.06,8137.29,8.4045,0.03,391,2388,100.0,38.76,23.4479


In [4]:
test.shape

(13096, 26)

In [5]:
test_life = test.groupby('unit')["cycle"].max()

In [6]:
test_life.describe()

count    100.000000
mean     130.960000
std       53.593479
min       31.000000
25%       88.750000
50%      133.500000
75%      164.250000
max      303.000000
Name: cycle, dtype: float64

In [7]:
rul = pd.read_csv(RAW / "RUL_FD001.txt", sep=r"\s+", header=None, names=["rul"])

In [8]:
rul

,rul
0,112
1,98
2,69
3,82
4,91
...,...
95,137
96,82
97,59
98,117


In [9]:
print(rul.shape)
print(rul.head)

(100, 1)
<bound method NDFrame.head of     rul
0   112
1    98
2    69
3    82
4    91
..  ...
95  137
96   82
97   59
98  117
99   20

[100 rows x 1 columns]>


In [10]:
rul["unit"] = range(1, len(rul) + 1)
rul.head

<bound method NDFrame.head of     rul  unit
0   112     1
1    98     2
2    69     3
3    82     4
4    91     5
..  ...   ...
95  137    96
96   82    97
97   59    98
98  117    99
99   20   100

[100 rows x 2 columns]>

In [11]:
from turbofan.features import add_rolling_features

test_feat = add_rolling_features(test)
last_rows = test_feat.groupby("unit").tail(1)

In [12]:
print(last_rows.shape)
print(last_rows[["unit", "cycle", "sensor_11", "sensor_11_mean"]].head())

(100, 56)
     unit  cycle  sensor_11  sensor_11_mean
30      1     31      47.23          47.288
79      2     49      47.67          47.509
205     3    126      47.88          47.705
311     4    106      47.65          47.630
409     5     98      47.46          47.533


In [13]:
test_final = last_rows.merge(rul, on="unit")
print(test_final[["unit", "cycle", "rul"]].head())

   unit  cycle  rul
0     1     31  112
1     2     49   98
2     3    126   69
3     4    106   82
4     5     98   91


In [14]:
print(test_final.shape)

(100, 57)


In [15]:
test_final.head()

,unit,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_14_std,sensor_15_mean,sensor_15_std,sensor_17_mean,sensor_17_std,sensor_20_mean,sensor_20_std,sensor_21_mean,sensor_21_std,rul
0,1,31,-0.0006,0.0004,100.0,518.67,642.58,1581.22,1398.91,14.62,...,2.631744,8.42013,0.022340,392.2,0.788811,38.933,0.114120,23.36944,0.026390,112
1,2,49,0.0018,-0.0001,100.0,518.67,642.55,1586.59,1410.83,14.62,...,2.054952,8.44631,0.023850,392.4,0.843274,38.847,0.084859,23.29325,0.049705,98
2,3,126,-0.0016,0.0004,100.0,518.67,642.88,1589.75,1418.89,14.62,...,2.904323,8.46210,0.022946,393.9,0.994429,38.743,0.134002,23.22671,0.039177,69
3,4,106,0.0012,0.0004,100.0,518.67,642.78,1594.53,1406.88,14.62,...,2.720683,8.45089,0.016581,393.4,1.074968,38.783,0.102637,23.26245,0.054676,82
4,5,98,-0.0013,-0.0004,100.0,518.67,642.27,1589.94,1419.36,14.62,...,3.669963,8.45198,0.018150,393.5,0.707107,38.774,0.082219,23.25765,0.078644,91


Loading the trained model back

In [16]:
import json
import joblib
from turbofan.data import load_test_rul
from turbofan.features import last_cycle_per_engine

In [17]:
model = joblib.load("../models/model.joblib")
features = json.loads(Path("../models/features.json").read_text())

latest = last_cycle_per_engine(add_rolling_features(test))
preds = model.predict(latest[features])

print("Engine 1 predicted RUL:", round(preds[0], 1))
print("Engine 1 true RUL:     ", load_test_rul(RAW / "RUL_FD001.txt")["rul"].iloc[0])

Engine 1 predicted RUL: 121.9
Engine 1 true RUL:      112


In [18]:
text = Path("../models/features.json").read_text()
print(type(text), text[:40])           # a string, starting with '['

features = json.loads(text)
print(type(features), len(features))   # a list, 45 items

with_features = add_rolling_features(test)
latest = last_cycle_per_engine(with_features)
print(with_features.shape, latest.shape)   # thousands of rows → 100 rows

X = latest[features]
print(X.shape)                          # (100, 45)

preds = model.predict(X)
print(preds.shape, preds[:5])           # 100 predictions; first five shown

<class 'str'> [
  "sensor_2",
  "sensor_3",
  "sensor_
<class 'list'> 45
(13096, 56) (100, 56)
(100, 45)
(100,) [121.92 122.14  50.91  94.33 105.28]
